# Week 8 — Regularization, Hyperparameter Tuning & Optimization
## Notebook 2: L1 Regularization — Lasso & Sparsity

**Difficulty:** ⭐⭐⭐⭐ &nbsp;&nbsp;|&nbsp;&nbsp; **Estimated Time:** 3 hours &nbsp;&nbsp;|&nbsp;&nbsp; **Theme:** California-Style House Price Prediction

---

## Real-World Analogy First

You're packing for a two-week trip and you're a chronic over-packer. Your suitcase is bursting with 50 items, but honestly only about 5 of them you'll actually use.

**Ridge Regression (L2) approach:**  
A strict airline weight limit. You keep all 50 items but replace your full-size toiletries with travel sizes. The hairdryer gets smaller, the shoes get lighter — everything is still there, just reduced. You still overpack 45 useless things.

**Lasso (L1) approach:**  
Marie Kondo arrives. She holds up each item and asks: "Does this spark joy? Will you actually use this?" The 15 formal shirts, the snow boots (going to Hawaii), the 3 backup cameras — gone. She doesn't make them smaller. She leaves them behind entirely. You end up with only the 5 items that actually matter.

> **Lasso is the Marie Kondo of machine learning: it doesn't shrink irrelevant features — it eliminates them entirely. This is the magic of L1 regularization.**

---

## Why Lasso Matters

In real ML problems:
- A house price dataset might have 100+ features, but maybe only 8–10 actually matter
- In genomics: 20,000 genes measured, perhaps 30 relevant to disease risk
- In NLP text features: millions of word n-grams, but a handful drive sentiment

Lasso handles **p >> n settings** (more features than samples) where ordinary least squares fails entirely. It performs automatic feature selection as part of the optimization process — no separate selection step needed.

---

## Learning Objectives

1. Derive and implement the L1 loss objective from scratch
2. Implement coordinate descent with soft-thresholding
3. Understand geometrically why L1 creates sparsity but L2 does not
4. Use Lasso as a feature selection tool and evaluate precision/recall
5. Choose the optimal regularization strength α using a validation curve

## Section 1: Setup

We'll use a 20-feature house price dataset where only 5 features are real signals — the other 15 are pure noise. Lasso should correctly zero out those 15 noise features.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyArrowPatch
import seaborn as sns
from sklearn.linear_model import Lasso, LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
sns.set_theme(style='whitegrid')

COLORS = {
    'real': '#2196F3',       # blue — real features
    'noise': '#F44336',      # red — noise features
    'train': '#4CAF50',      # green
    'val': '#FF9800',        # orange
    'lasso': '#9C27B0',      # purple
    'ridge': '#00BCD4',      # cyan
    'zero': '#9E9E9E',       # grey
}

print('All libraries loaded.')

In [ ]:
# ---------------------------------------------------------------------------
# Dataset: 300 samples, 20 features
#   5 REAL features (have genuine relationship with price)
#   15 NOISE features (pure random numbers, no relationship with price)
# ---------------------------------------------------------------------------
np.random.seed(42)
N = 300
N_REAL  = 5
N_NOISE = 15
N_FEAT  = N_REAL + N_NOISE  # 20 total

# Real feature names and their true coefficients
real_feature_names = ['sqft', 'bedrooms', 'age', 'dist_downtown', 'school_rating']
true_coefficients  = np.array([3.5, 0.8, -0.04, -0.6, 1.2])  # true betas

# Generate real features
sqft          = np.random.uniform(600, 4000, N)
bedrooms      = np.random.randint(1, 6, N).astype(float)
age           = np.random.randint(1, 60, N).astype(float)
dist_downtown = np.random.uniform(0, 30, N)   # miles from downtown
school_rating = np.random.uniform(1, 10, N)   # school district rating

X_real = np.column_stack([sqft, bedrooms, age, dist_downtown, school_rating])

# Generate 15 noise features (completely uncorrelated with price)
noise_feature_names = [f'noise_{i:02d}' for i in range(1, N_NOISE + 1)]
X_noise = np.random.randn(N, N_NOISE) * np.random.uniform(0.5, 5, N_NOISE)

# Combine
X_raw = np.column_stack([X_real, X_noise])
feature_names = real_feature_names + noise_feature_names

# True price (only real features contribute)
price_true = 200 + X_real @ true_coefficients   # base 200k + linear combination
noise_eps  = np.random.normal(0, 15, N)          # Gaussian noise (σ=$15k)
y          = price_true + noise_eps

# Scale features (important for regularization — ensures fair penalty)
scaler = StandardScaler()
X      = scaler.fit_transform(X_raw)

# Train / validation split
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.25, random_state=42
)

# Ground truth feature mask
is_real = np.array([True]*N_REAL + [False]*N_NOISE)

print(f'Dataset: {N} samples, {N_FEAT} features ({N_REAL} real, {N_NOISE} noise)')
print(f'Train: {X_train.shape}  |  Val: {X_val.shape}')
print(f'Price range: ${y.min():.0f}k — ${y.max():.0f}k  (mean=${y.mean():.0f}k)')
print()
print('True coefficients (standardized features):')
for name, coef in zip(real_feature_names, true_coefficients):
    print(f'  {name:20s}: {coef:+.2f}')
print('  noise features      : 0.00 (×15)')

In [ ]:
# ---------------------------------------------------------------------------
# Section 1b: Exploratory Data Analysis — Feature Distributions
# Before any modeling, always inspect the data. Verify that noise features
# have no obvious visual relationship with price (they should look like noise).
# ---------------------------------------------------------------------------
fig, axes = plt.subplots(2, 5, figsize=(18, 7))
axes = axes.flatten()

# Show all 5 real features and 5 sample noise features
show_features = real_feature_names + [f'noise_{i:02d}' for i in range(1, 6)]

for ax, feat in zip(axes, show_features):
    col_idx = feature_names.index(feat)
    x_raw = X_raw[:, col_idx]  # unscaled
    is_r  = feat in real_feature_names

    ax.scatter(x_raw, y, alpha=0.3, s=15,
               color=COLORS['real'] if is_r else COLORS['noise'],
               edgecolors='none')

    # Simple linear trend line
    coef = np.polyfit(x_raw, y, 1)
    x_line = np.linspace(x_raw.min(), x_raw.max(), 100)
    ax.plot(x_line, np.polyval(coef, x_line),
            color='black', linewidth=1.8, linestyle='--', alpha=0.7)

    # Pearson correlation
    r = np.corrcoef(x_raw, y)[0, 1]
    label_type = 'REAL' if is_r else 'NOISE'
    ax.set_title(f'{feat}\nr = {r:.3f}  [{label_type}]',
                 fontsize=10, fontweight='bold',
                 color='darkblue' if is_r else 'darkred')
    ax.set_xlabel(feat, fontsize=9)
    ax.set_ylabel('Price ($k)', fontsize=9)

plt.suptitle('Feature vs. Price: Real Features Show Clear Correlation; Noise Features Do Not\n'
             'Blue = real (signal), Red = noise (no correlation expected)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Correlation table
corr_vals = [(feat, np.corrcoef(X_raw[:, i], y)[0, 1], feat in real_feature_names)
             for i, feat in enumerate(feature_names)]
corr_vals.sort(key=lambda x: abs(x[1]), reverse=True)
print('Top 10 feature correlations with price:')
print(f'{"Feature":<22} {"Pearson r":>10}  {"Type":<8}')
print('-' * 44)
for feat, r, is_r in corr_vals[:10]:
    print(f'{feat:<22} {r:>10.4f}  {"REAL" if is_r else "NOISE"}')

## Section 2: The L1 Objective Function

### Ordinary Least Squares (OLS)
$$\min_{\theta} \; \frac{1}{2n} \sum_{i=1}^{n}(y_i - \hat{y}_i)^2$$

OLS finds coefficients that minimize residual sum of squares with **no constraint** on the size of the coefficients. This leads to overfitting when there are many features.

### Lasso (L1 Regularization)
$$\min_{\theta} \; \underbrace{\frac{1}{2n} \sum_{i=1}^{n}(y_i - \hat{y}_i)^2}_{\text{data fit term}} + \underbrace{\alpha \sum_{j=1}^{p}|\theta_j|}_{\text{L1 penalty}}$$

**What each part does:**
- **Data fit term:** Same as OLS — measures how well the model fits the training data
- **L1 penalty:** Penalizes the absolute value of each coefficient. Large coefficients are expensive.

**The α parameter:**
- α = 0 → No penalty → same as OLS (can overfit)
- α → ∞ → Maximum penalty → all coefficients driven to zero (extreme underfitting)
- Optimal α → somewhere in between (tune via cross-validation)

### Why Does L1 Create Exactly-Zero Coefficients?
The L1 penalty |θ| is **not differentiable at θ = 0**. At that kink point, many subgradients pass through zero simultaneously — meaning the gradient condition is satisfied without the coefficient needing to move away from zero. This creates a **dead zone** around zero that "traps" small coefficients.

L2 penalty θ² is differentiable everywhere — its gradient at θ = 0 is 0, so it doesn't push coefficients to exactly zero, just near zero.

## Section 3: Coordinate Descent — The Lasso Algorithm

Gradient descent cannot be used directly for Lasso because the L1 penalty is not differentiable at zero. Instead, we use **coordinate descent**: update one coefficient at a time while holding all others fixed.

### Update Rule for Coordinate $j$

**Step 1:** Compute partial residual — what the data looks like when we remove feature $j$'s contribution:
$$r_j = y - X\theta + X_{:,j}\theta_j$$

**Step 2:** Compute the unconstrained optimal coefficient for feature $j$:
$$\rho_j = \frac{1}{n} X_{:,j}^T r_j$$

**Step 3:** Apply soft-thresholding (this is where the L1 magic happens):
$$\theta_j \leftarrow S(\rho_j / z_j, \; \alpha / z_j) \quad \text{where} \quad z_j = \frac{1}{n}\sum_i X_{ij}^2$$

### The Soft-Thresholding Operator $S(\rho, \alpha)$
$$S(\rho, \alpha) = \begin{cases} \rho + \alpha & \text{if } \rho < -\alpha \\ 0 & \text{if } |\rho| \leq \alpha \\ \rho - \alpha & \text{if } \rho > \alpha \end{cases}$$

The **dead zone** $|\rho| \leq \alpha$ is precisely what creates sparsity: if the unconstrained optimal coefficient is small enough, it gets set exactly to zero.

In [ ]:
# ---------------------------------------------------------------------------
# Section 3: Implement Lasso from Scratch
# Coordinate descent with soft-thresholding.
# ---------------------------------------------------------------------------

def soft_threshold(rho: float, alpha: float) -> float:
    """
    Soft-thresholding operator S(rho, alpha).

    This is the closed-form solution to:
        argmin_{theta} 0.5*(theta - rho)^2 + alpha*|theta|

    The dead zone [−alpha, +alpha] sets small values exactly to zero.
    This is the key to Lasso sparsity.
    """
    if rho < -alpha:
        return rho + alpha   # negative side: shift right
    elif rho > alpha:
        return rho - alpha   # positive side: shift left
    else:
        return 0.0           # dead zone: zero it out


def lasso_coordinate_descent(
    X: np.ndarray,
    y: np.ndarray,
    alpha: float = 1.0,
    max_iter: int = 1000,
    tol: float = 1e-4
) -> np.ndarray:
    """
    Fit Lasso regression via coordinate descent.

    Parameters:
    -----------
    X       : (n, p) feature matrix (should be standardized)
    y       : (n,) target vector
    alpha   : regularization strength
    max_iter: maximum number of full coordinate passes
    tol     : convergence tolerance (max coefficient change)

    Returns:
    --------
    theta : (p,) coefficient vector
    """
    n, p = X.shape
    theta = np.zeros(p)  # initialize all coefficients to zero

    # Precompute z_j = mean(X_j^2) — the denominator for each coordinate
    # For standardized X with zero mean, z_j ≈ 1
    z = (X**2).mean(axis=0)  # shape (p,)

    for iteration in range(max_iter):
        theta_old = theta.copy()

        for j in range(p):
            # Partial residual: remove feature j's contribution
            # r_j = y - X @ theta + X[:,j] * theta[j]
            r_j   = y - (X @ theta) + X[:, j] * theta[j]

            # Correlation of feature j with partial residual
            rho_j = (X[:, j] @ r_j) / n

            # Soft-threshold: this is the per-coordinate optimal update
            theta[j] = soft_threshold(rho_j / z[j], alpha / z[j])

        # Check convergence: max absolute change in coefficients
        if np.max(np.abs(theta - theta_old)) < tol:
            # print(f'  Converged at iteration {iteration+1}')
            break

    return theta


print('Soft-thresholding function defined.')
print('Coordinate descent Lasso implemented.')
print()
print('Quick test of soft_threshold:')
test_rhos = [-2.0, -0.5, 0.0, 0.5, 2.0]
alpha_test = 1.0
for rho in test_rhos:
    result = soft_threshold(rho, alpha_test)
    zone = 'DEAD ZONE' if abs(rho) <= alpha_test else 'ACTIVE'
    print(f'  S({rho:+.1f}, α={alpha_test}) = {result:+.1f}  [{zone}]')

## Section 4b: OLS Baseline — Why We Need Regularization Here

Before diving into Lasso, let's see what happens when we naively run OLS on all 20 features including the 15 noise features. With 300 samples and 20 features, OLS is technically possible — but the noise features inflate variance and hurt generalization.

In [ ]:
# ---------------------------------------------------------------------------
# Section 4b: OLS Baseline on All 20 Features
# Shows that OLS fits noise features and has poor generalization vs.
# OLS on only the 5 true features.
# ---------------------------------------------------------------------------
# OLS on all 20 features (including 15 noise)
ols_all = LinearRegression(fit_intercept=True)
ols_all.fit(X_train, y_train)

# OLS on only the 5 real features (oracle — cheating, but illustrates ideal)
ols_real_only = LinearRegression(fit_intercept=True)
ols_real_only.fit(X_train[:, :5], y_train)

train_mse_all  = mean_squared_error(y_train, ols_all.predict(X_train))
val_mse_all    = mean_squared_error(y_val,   ols_all.predict(X_val))
train_mse_real = mean_squared_error(y_train, ols_real_only.predict(X_train[:, :5]))
val_mse_real   = mean_squared_error(y_val,   ols_real_only.predict(X_val[:, :5]))

print('OLS Performance Comparison:')
print(f'{"Model":<30} {"Train RMSE":>12} {"Val RMSE":>12} {"Gap":>10}')
print('-' * 68)
for name, tr, vl in [
    ('OLS — all 20 features',      train_mse_all**0.5,  val_mse_all**0.5),
    ('OLS — 5 real features only', train_mse_real**0.5, val_mse_real**0.5),
]:
    gap = vl - tr
    print(f'{name:<30} {tr:>12.2f} {vl:>12.2f} {gap:>10.2f}')

print()
print('Observation: OLS on all 20 features has higher validation error than')
print('OLS on just the 5 real features — the 15 noise features add variance!')

# Visualize OLS coefficients: how much does OLS give to noise features?
fig, ax = plt.subplots(figsize=(13, 4))
coefs_ols = ols_all.coef_
bar_cols = [COLORS['real']]*5 + [COLORS['noise']]*15
bars = ax.bar(range(len(coefs_ols)), coefs_ols, color=bar_cols, edgecolor='white', alpha=0.85)
ax.axhline(0, color='black', linewidth=1.0)
ax.axvline(4.5, color='black', linewidth=2, linestyle='--', alpha=0.6)
ax.text(1.5,  max(coefs_ols)*0.88, 'REAL\nFEATURES', ha='center', fontsize=11,
        color='darkblue', fontweight='bold')
ax.text(12.0, max(coefs_ols)*0.88, 'NOISE FEATURES (should all be ≈ 0)',
        ha='center', fontsize=11, color='darkred', fontweight='bold')
real_patch  = mpatches.Patch(color=COLORS['real'],  label='Real features')
noise_patch = mpatches.Patch(color=COLORS['noise'], label='Noise features')
ax.legend(handles=[real_patch, noise_patch], fontsize=11)
ax.set_xlabel('Feature Index', fontsize=12)
ax.set_ylabel('OLS Coefficient', fontsize=12)
ax.set_title('OLS Coefficients — Noise Features Get Non-Zero Weights (They Should Be 0)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# Section 4: Verify — Scratch Lasso vs sklearn Lasso
# ---------------------------------------------------------------------------
ALPHA_TEST = 0.5

# Our implementation
theta_scratch = lasso_coordinate_descent(X_train, y_train, alpha=ALPHA_TEST,
                                          max_iter=2000, tol=1e-6)

# sklearn implementation
lasso_sk = Lasso(alpha=ALPHA_TEST, fit_intercept=False, max_iter=10000, tol=1e-6)
lasso_sk.fit(X_train, y_train)
theta_sklearn = lasso_sk.coef_

# Comparison
df_compare = pd.DataFrame({
    'Feature': feature_names,
    'Scratch': theta_scratch,
    'sklearn': theta_sklearn,
    'Diff': np.abs(theta_scratch - theta_sklearn),
    'Is_Real': is_real
})

print(f'Lasso Coefficient Comparison (α={ALPHA_TEST})')
print(f'{"Feature":<22} {"Scratch":>10} {"sklearn":>10} {"|Diff|":>10}')
print('-' * 55)
for _, row in df_compare.iterrows():
    marker = '★' if row['Is_Real'] else ' '
    print(f'{marker} {row["Feature"]:<20} {row["Scratch"]:>10.4f} {row["sklearn"]:>10.4f} {row["Diff"]:>10.6f}')

max_diff = df_compare['Diff'].max()
print(f'\nMax absolute difference: {max_diff:.6f}')
print(f'Agreement: {"EXCELLENT (< 0.01)" if max_diff < 0.01 else "CHECK IMPLEMENTATION"}')
print('★ = real feature')

In [ ]:
# ---------------------------------------------------------------------------
# Section 5: Soft-Thresholding Visualization
# Plot the function shape: input rho vs output theta
# ---------------------------------------------------------------------------
rho_vals = np.linspace(-3.5, 3.5, 500)
alpha_vals_vis = [0.5, 1.0, 1.5]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Soft-thresholding for multiple alpha
ax = axes[0]
colors_alpha = ['#2196F3', '#F44336', '#4CAF50']
for a, col in zip(alpha_vals_vis, colors_alpha):
    theta_vals = np.array([soft_threshold(r, a) for r in rho_vals])
    ax.plot(rho_vals, theta_vals, color=col, linewidth=2.5, label=f'α = {a}')

ax.plot(rho_vals, rho_vals, 'k--', linewidth=1.5, alpha=0.5, label='Identity (no penalty)')
ax.axhline(0, color='black', linewidth=0.8)
ax.axvline(0, color='black', linewidth=0.8)

# Annotate dead zone for alpha=1
ax.axvspan(-1.0, 1.0, alpha=0.08, color='red', label='Dead zone (α=1.0)')
ax.annotate('Dead Zone\n(output = 0)', xy=(0, 0), xytext=(1.5, -1.5),
            arrowprops=dict(arrowstyle='->', color='red', lw=1.5),
            fontsize=11, color='red')

ax.set_xlabel('ρ (unconstrained optimal coefficient)', fontsize=12)
ax.set_ylabel('θ (Lasso output coefficient)', fontsize=12)
ax.set_title('Soft-Thresholding Operator S(ρ, α)\n'
             'Values inside the dead zone become exactly 0', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)

# Plot 2: Compare soft-threshold vs hard-threshold vs identity
ax = axes[1]
alpha_compare = 1.0

soft_vals = np.array([soft_threshold(r, alpha_compare) for r in rho_vals])
hard_vals = np.where(np.abs(rho_vals) > alpha_compare, rho_vals, 0.0)
ridge_vals = rho_vals / (1 + alpha_compare)  # Ridge shrinkage (proportional)

ax.plot(rho_vals, soft_vals, color='#9C27B0', linewidth=2.5, label='Lasso (soft-threshold)')
ax.plot(rho_vals, hard_vals, color='#F44336', linewidth=2.5, linestyle='--', label='Hard threshold')
ax.plot(rho_vals, ridge_vals, color='#00BCD4', linewidth=2.5, linestyle=':', label='Ridge (proportional shrink)')
ax.plot(rho_vals, rho_vals, 'k--', linewidth=1.2, alpha=0.4, label='No regularization')

ax.axhline(0, color='black', linewidth=0.8)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('ρ (unconstrained)', fontsize=12)
ax.set_ylabel('θ (regularized output)', fontsize=12)
ax.set_title('Lasso vs Ridge vs Hard Threshold\n'
             'Only Lasso creates true zeros; Ridge always keeps non-zero values', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)

plt.suptitle('The Soft-Thresholding Operator: Heart of Lasso', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# Section 6: L1 vs L2 Constraint Geometry
# Diamond (L1) vs Circle (L2) — Why L1 corners create sparsity
# ---------------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

theta_range = np.linspace(-2, 2, 400)

for ax, title, shape_label, color in zip(
    axes,
    ['L1 Constraint (Lasso): Diamond\nCorners on axes → creates zeros',
     'L2 Constraint (Ridge): Circle\nSmooth boundary → near-zero but not zero'],
    ['L1 Ball', 'L2 Ball'],
    [COLORS['lasso'], COLORS['ridge']]
):
    # Constraint region
    t = np.linspace(0, 2*np.pi, 500)
    if shape_label == 'L1 Ball':  # diamond: |θ1| + |θ2| ≤ 1
        theta1_c = np.sign(np.cos(t)) * (1 - np.abs(np.sin(t)))
        theta2_c = np.sign(np.sin(t)) * (1 - np.abs(np.cos(t)))
    else:  # circle: θ1² + θ2² ≤ 1
        theta1_c = np.cos(t)
        theta2_c = np.sin(t)

    ax.fill(theta1_c, theta2_c, alpha=0.2, color=color, label=shape_label)
    ax.plot(theta1_c, theta2_c, color=color, linewidth=2.5)

    # OLS solution (elliptical contours — we approximate as a single point)
    ols_theta1, ols_theta2 = 1.5, 1.2

    # Draw OLS error ellipses (contours of the loss function)
    for scale, alpha_ell in [(0.4, 0.35), (0.7, 0.25), (1.05, 0.18)]:
        ell_t1 = ols_theta1 + scale * np.cos(t) * 0.8
        ell_t2 = ols_theta2 + scale * np.sin(t) * 0.6
        ax.plot(ell_t1, ell_t2, color='#607D8B', linewidth=1.2, alpha=alpha_ell)

    ax.scatter([ols_theta1], [ols_theta2], s=120, color='black',
               zorder=5, label='OLS solution (unconstrained)')

    # Lasso / Ridge solution: where ellipse first touches the constraint region
    if shape_label == 'L1 Ball':
        # Touches the corner at (1, 0) — θ2 = 0 → sparsity!
        contact1, contact2 = 1.0, 0.0
        ax.scatter([contact1], [contact2], s=180, color=color, zorder=6,
                   marker='*', label='Lasso solution (θ₂ = 0 exactly!)')
        ax.annotate('θ₂ = 0\n(EXACT ZERO)', xy=(contact1, contact2),
                    xytext=(0.3, -0.7),
                    arrowprops=dict(arrowstyle='->', color='darkred', lw=1.5),
                    fontsize=11, color='darkred', fontweight='bold')
        # Mark corners
        for cx, cy, cname in [(1,0,'(1,0)'),(0,1,'(0,1)'),(-1,0,'(-1,0)'),(0,-1,'(0,-1)')]:
            ax.scatter([cx], [cy], s=60, color='darkred', zorder=4)
        ax.text(0, 1.15, 'Corners on axes\n= one coef is ZERO', ha='center',
                fontsize=9, color='darkred')
    else:
        # Touches the circle at a point away from axes — no sparsity
        # Approximate contact point
        contact1, contact2 = 0.78, 0.62
        ax.scatter([contact1], [contact2], s=180, color=color, zorder=6,
                   marker='*', label='Ridge solution (both θ non-zero)')
        ax.annotate('Both θ₁, θ₂ > 0\n(no exact zeros)', xy=(contact1, contact2),
                    xytext=(-0.5, 0.8),
                    arrowprops=dict(arrowstyle='->', color='darkblue', lw=1.5),
                    fontsize=11, color='darkblue', fontweight='bold')

    # Axes
    ax.axhline(0, color='black', linewidth=0.8, alpha=0.6)
    ax.axvline(0, color='black', linewidth=0.8, alpha=0.6)
    ax.set_xlim(-2.2, 2.2)
    ax.set_ylim(-1.8, 1.8)
    ax.set_xlabel('θ₁ (Feature 1 coefficient)', fontsize=12)
    ax.set_ylabel('θ₂ (Feature 2 coefficient)', fontsize=12)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.legend(fontsize=9, loc='upper left')
    ax.set_aspect('equal')

plt.suptitle('Why L1 Creates Sparsity but L2 Does Not:\nConstraint Region Geometry',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('Key geometric insight:')
print('  L1 (diamond): Has corners on the coordinate axes.')
print('  When the OLS ellipse expands, it almost always hits a CORNER first.')
print('  At a corner, one coordinate = 0 → sparsity.')
print()
print('  L2 (circle): Smooth boundary, no corners.')
print('  Ellipse touches the circle at a generic point away from axes.')
print('  Both coordinates are non-zero (just small).')

In [ ]:
# ---------------------------------------------------------------------------
# Section 7: Sparsity Demo — Coefficients as Alpha Increases
# As α grows, more coefficients hit zero.
# We expect the 15 noise features to zero out first.
# ---------------------------------------------------------------------------
alphas = [0.001, 0.01, 0.05, 0.1, 0.5, 1.0, 5.0, 10.0]

coef_matrix = np.zeros((len(alphas), N_FEAT))
n_nonzero   = []
val_mses    = []
train_mses  = []

for i, alpha in enumerate(alphas):
    lasso = Lasso(alpha=alpha, fit_intercept=True, max_iter=10000, tol=1e-8)
    lasso.fit(X_train, y_train)
    coef_matrix[i] = lasso.coef_
    n_nonzero.append(np.sum(np.abs(lasso.coef_) > 1e-6))
    val_mses.append(mean_squared_error(y_val, lasso.predict(X_val)))
    train_mses.append(mean_squared_error(y_train, lasso.predict(X_train)))

print('Number of non-zero coefficients by alpha:')
print(f'{"Alpha":<12} {"Non-zero":>10} {"Train MSE":>12} {"Val MSE":>12}')
print('-' * 50)
for a, nz, tr, vl in zip(alphas, n_nonzero, train_mses, val_mses):
    print(f'{a:<12.4g} {nz:>10} {tr:>12.2f} {vl:>12.2f}')

In [ ]:
# ---------------------------------------------------------------------------
# Section 7b: Coefficient Path (Regularization Path)
# Shows how each coefficient changes as alpha varies.
# ---------------------------------------------------------------------------
fig, axes = plt.subplots(2, 1, figsize=(13, 11))

# Plot 1: Full coefficient path
ax = axes[0]
alpha_log = np.log10(alphas)

for j in range(N_FEAT):
    if is_real[j]:
        ax.plot(alpha_log, coef_matrix[:, j],
                linewidth=2.5, color=COLORS['real'],
                label=feature_names[j] if j < N_REAL else None,
                solid_capstyle='round')
    else:
        ax.plot(alpha_log, coef_matrix[:, j],
                linewidth=1.0, color=COLORS['noise'], alpha=0.4,
                linestyle='--')

# Labels for real features at left edge
for j, name in enumerate(real_feature_names):
    y_pos = coef_matrix[0, j]
    ax.text(alpha_log[0] - 0.05, y_pos, name, fontsize=9,
            color=COLORS['real'], ha='right', va='center')

ax.axhline(0, color='black', linewidth=1.2, linestyle='--', alpha=0.6)
real_patch  = mpatches.Patch(color=COLORS['real'],  label='Real features (5)')
noise_patch = mpatches.Patch(color=COLORS['noise'], label='Noise features (15)')
ax.legend(handles=[real_patch, noise_patch], fontsize=11, loc='upper right')
ax.set_xlabel('log₁₀(α) — Regularization Strength →', fontsize=12)
ax.set_ylabel('Coefficient Value', fontsize=12)
ax.set_title('Lasso Regularization Path: Coefficients vs. α\n'
             'Real features (blue) persist longer; noise (red) zeroed out first', fontsize=13, fontweight='bold')

# Plot 2: Number of non-zero coefficients
ax = axes[1]
bar_colors = [COLORS['real'] if nz <= N_REAL else
              (COLORS['noise'] if nz > 10 else '#FF9800')
              for nz in n_nonzero]
bars = ax.bar(range(len(alphas)), n_nonzero, color=bar_colors, edgecolor='white', linewidth=1.5)
for bar, nz in zip(bars, n_nonzero):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
            str(int(nz)), ha='center', fontsize=12, fontweight='bold')
ax.axhline(N_REAL, color='darkblue', linestyle='--', linewidth=2,
           label=f'True # relevant features = {N_REAL}')
ax.set_xticks(range(len(alphas)))
ax.set_xticklabels([f'α={a}' for a in alphas], rotation=30, fontsize=10)
ax.set_ylabel('Number of Non-Zero Coefficients', fontsize=12)
ax.set_title('Sparsity: How Many Features Does Lasso Keep?\n'
             'Blue = found the true 5; red = too many; higher → sparser', fontsize=12, fontweight='bold')
ax.legend(fontsize=11)

plt.tight_layout()
plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# Section 8: Grouped Bar Chart — Coefficients at Multiple Alphas
# Show how individual feature coefficients change with regularization.
# ---------------------------------------------------------------------------
showcase_alphas = [0.001, 0.1, 1.0, 10.0]
coef_showcase = {}

for alpha in showcase_alphas:
    lasso = Lasso(alpha=alpha, fit_intercept=True, max_iter=10000, tol=1e-8)
    lasso.fit(X_train, y_train)
    coef_showcase[alpha] = lasso.coef_

# Plot only first 10 features for clarity (5 real + 5 noise)
show_idx  = list(range(N_REAL)) + list(range(N_REAL, N_REAL + 5))
show_names = [feature_names[i] for i in show_idx]

fig, ax = plt.subplots(figsize=(14, 6))
x = np.arange(len(show_idx))
width = 0.2
bar_colors_alpha = ['#1565C0', '#42A5F5', '#EF9A9A', '#B71C1C']

for k, (alpha, col) in enumerate(zip(showcase_alphas, bar_colors_alpha)):
    vals = [coef_showcase[alpha][i] for i in show_idx]
    bars = ax.bar(x + k*width - 1.5*width, vals,
                  width, label=f'α={alpha}', color=col, alpha=0.85, edgecolor='white')

ax.axhline(0, color='black', linewidth=1.2)

# Vertical separator between real and noise features
ax.axvline(N_REAL - 0.5, color='black', linewidth=2.0, linestyle='--', alpha=0.6)
ax.text(1.5, ax.get_ylim()[1] * 0.82 if ax.get_ylim()[1] > 0 else 2.5,
        'REAL FEATURES', ha='center', fontsize=11, color='darkblue', fontweight='bold')
ax.text(N_REAL + 1.5, ax.get_ylim()[1] * 0.82 if ax.get_ylim()[1] > 0 else 2.5,
        'NOISE FEATURES', ha='center', fontsize=11, color='darkred', fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(show_names, rotation=25, fontsize=11)
ax.set_ylabel('Coefficient Value', fontsize=12)
ax.set_title('Lasso Coefficients at Different α Values\n'
             'Higher α → more coefficients collapse to exactly zero', fontsize=13, fontweight='bold')
ax.legend(fontsize=11, title='Regularization strength')
plt.tight_layout()
plt.show()

print('Observation: At α=1.0 and above, all noise features are exactly 0.')
print('Real features persist longer because they have genuine predictive signal.')

## Section 11b: LassoCV — Automated Optimal Alpha via Cross-Validation

Rather than manually sweeping alpha values, sklearn's `LassoCV` automatically selects the best alpha using cross-validation. It is the standard production-ready way to fit Lasso.

In [ ]:
# ---------------------------------------------------------------------------
# Section 11b: LassoCV — Find the Best Alpha Automatically
# Uses 5-fold cross-validation internally across a grid of alpha values.
# ---------------------------------------------------------------------------
from sklearn.linear_model import LassoCV
from sklearn.model_selection import KFold

np.random.seed(42)
kf = KFold(n_splits=5, shuffle=True, random_state=42)

lasso_cv = LassoCV(
    alphas=np.logspace(-3, 2, 80),
    cv=kf,
    fit_intercept=True,
    max_iter=20000,
    random_state=42
)
lasso_cv.fit(X_train, y_train)

print(f'LassoCV selected alpha: {lasso_cv.alpha_:.5f}')
print(f'Number of non-zero coefficients: {np.sum(np.abs(lasso_cv.coef_) > 1e-6)} / {X_train.shape[1]}')
print()

# Which features were selected?
selected_mask = np.abs(lasso_cv.coef_) > 1e-6
print('Features selected by LassoCV:')
for i, (name, coef, sel) in enumerate(zip(feature_names, lasso_cv.coef_, selected_mask)):
    truth = 'REAL' if is_real[i] else 'NOISE'
    status = 'SELECTED' if sel else 'zeroed out'
    marker = '★' if (sel and is_real[i]) else ('✗' if (sel and not is_real[i]) else ' ')
    print(f'  {marker} {name:<22}: {coef:+8.4f}  [{truth}] → {status}')

# Evaluate on validation set
val_pred_cv = lasso_cv.predict(X_val)
val_rmse_cv = mean_squared_error(y_val, val_pred_cv)**0.5
print(f'\nLassoCV validation RMSE: ${val_rmse_cv:.2f}k')

# Visualize the CV MSE path (what LassoCV computed internally)
cv_mse_path = lasso_cv.mse_path_.mean(axis=1)  # mean over folds

fig, ax = plt.subplots(figsize=(10, 5))
ax.semilogx(lasso_cv.alphas_, cv_mse_path, linewidth=2.5, color=COLORS['lasso'])
ax.axvline(lasso_cv.alpha_, color='darkgreen', linewidth=2.5, linestyle='--',
           label=f'Selected alpha = {lasso_cv.alpha_:.4f}')
ax.scatter([lasso_cv.alpha_], [cv_mse_path[np.argmin(cv_mse_path)]],
           s=200, color='darkgreen', zorder=6, marker='*')
ax.set_xlabel('Alpha (log scale)', fontsize=12)
ax.set_ylabel('Cross-Validation MSE', fontsize=12)
ax.set_title('LassoCV: Internal Alpha Selection via 5-Fold Cross-Validation\n'
             'The algorithm automatically found the optimal alpha',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# Section 9: Feature Selection Performance
# Compute Precision and Recall for identifying the 5 true features.
#
# Precision = of features Lasso selected, how many were actually real?
# Recall    = of the 5 real features, how many did Lasso find?
# F1        = harmonic mean of precision and recall
# ---------------------------------------------------------------------------
alphas_fine = np.logspace(-3, 1.5, 40)

precisions, recalls, f1s = [], [], []
n_selected = []

for alpha in alphas_fine:
    lasso = Lasso(alpha=alpha, fit_intercept=True, max_iter=10000, tol=1e-8)
    lasso.fit(X_train, y_train)
    selected = np.abs(lasso.coef_) > 1e-6  # predicted as relevant

    tp = np.sum(selected & is_real)      # true positives
    fp = np.sum(selected & ~is_real)     # false positives (selected noise)
    fn = np.sum(~selected & is_real)     # false negatives (missed real)

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

    precisions.append(precision)
    recalls.append(recall)
    f1s.append(f1)
    n_selected.append(np.sum(selected))

best_f1_idx = np.argmax(f1s)
best_alpha_f1 = alphas_fine[best_f1_idx]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Precision / Recall / F1 vs alpha
ax = axes[0]
ax.semilogx(alphas_fine, precisions, linewidth=2.5, color='#2196F3', label='Precision')
ax.semilogx(alphas_fine, recalls,    linewidth=2.5, color='#F44336', label='Recall')
ax.semilogx(alphas_fine, f1s,        linewidth=2.5, color='#4CAF50', label='F1 Score', linestyle='--')
ax.axvline(best_alpha_f1, color='black', linewidth=1.5, linestyle=':',
           label=f'Best F1 at α={best_alpha_f1:.3f}')
ax.set_xlabel('Regularization Strength α (log scale)', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Feature Selection Quality vs. α\n'
             'How well does Lasso identify the 5 true features?', fontsize=12, fontweight='bold')
ax.legend(fontsize=11)
ax.set_ylim([-0.05, 1.1])

# Plot 2: Number of selected features vs alpha
ax = axes[1]
ax.semilogx(alphas_fine, n_selected, linewidth=2.5, color=COLORS['lasso'], marker=None)
ax.fill_between(alphas_fine, n_selected, N_REAL, where=[n <= N_REAL for n in n_selected],
                alpha=0.15, color='blue', label='Under-selecting')
ax.fill_between(alphas_fine, n_selected, N_REAL, where=[n > N_REAL for n in n_selected],
                alpha=0.15, color='red', label='Over-selecting (noise included)')
ax.axhline(N_REAL, color='black', linewidth=2, linestyle='--', label=f'True # features = {N_REAL}')
ax.axvline(best_alpha_f1, color='green', linewidth=1.5, linestyle=':',
           label=f'Optimal α = {best_alpha_f1:.3f}')
ax.set_xlabel('Regularization Strength α (log scale)', fontsize=12)
ax.set_ylabel('Number of Selected Features', fontsize=12)
ax.set_title('How Many Features Does Lasso Select?\n'
             'Too small α: selects noise; Too large α: misses real features', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)

plt.suptitle('Lasso as Feature Selector: Precision, Recall, and Sparsity',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Best F1 score: {f1s[best_f1_idx]:.4f} at α = {best_alpha_f1:.4f}')
print(f'  Precision: {precisions[best_f1_idx]:.3f}  (of features selected, how many are real?)')
print(f'  Recall:    {recalls[best_f1_idx]:.3f}  (of real features, how many were found?)')

In [ ]:
# ---------------------------------------------------------------------------
# Section 10: MSE vs Alpha — Finding the Optimal Regularization Strength
# ---------------------------------------------------------------------------
alphas_mse = np.logspace(-3, 2, 60)
train_mse_list, val_mse_list = [], []

for alpha in alphas_mse:
    lasso = Lasso(alpha=alpha, fit_intercept=True, max_iter=10000, tol=1e-8)
    lasso.fit(X_train, y_train)
    train_mse_list.append(mean_squared_error(y_train, lasso.predict(X_train)))
    val_mse_list.append(mean_squared_error(y_val,   lasso.predict(X_val)))

best_val_idx   = np.argmin(val_mse_list)
best_alpha_mse = alphas_mse[best_val_idx]
best_val_mse   = val_mse_list[best_val_idx]

fig, ax = plt.subplots(figsize=(11, 6))

ax.semilogx(alphas_mse, train_mse_list, linewidth=2.5, color=COLORS['train'], label='Train MSE')
ax.semilogx(alphas_mse, val_mse_list,   linewidth=2.5, color=COLORS['val'],   label='Validation MSE')
ax.fill_between(alphas_mse, train_mse_list, val_mse_list, alpha=0.12, color='red',
                label='Generalization gap')

# Mark optimal
ax.axvline(best_alpha_mse, color='darkgreen', linewidth=2.5, linestyle='--',
           label=f'Optimal α = {best_alpha_mse:.4f}')
ax.scatter([best_alpha_mse], [best_val_mse], s=200, color='darkgreen', zorder=6,
           marker='*')
ax.annotate(f'Min Val MSE\nα = {best_alpha_mse:.3f}\nMSE = {best_val_mse:.1f}',
            xy=(best_alpha_mse, best_val_mse),
            xytext=(best_alpha_mse * 5, best_val_mse + (max(val_mse_list) - min(val_mse_list)) * 0.3),
            arrowprops=dict(arrowstyle='->', color='black', lw=1.5),
            fontsize=11, fontweight='bold')

# Zones
ax.axvspan(alphas_mse[0], best_alpha_mse, alpha=0.05, color='red', label='Under-regularized (overfitting)')
ax.axvspan(best_alpha_mse, alphas_mse[-1], alpha=0.05, color='blue', label='Over-regularized (underfitting)')

ax.set_xlabel('Regularization Strength α (log scale)', fontsize=13)
ax.set_ylabel('Mean Squared Error ($k²)', fontsize=13)
ax.set_title('Lasso: Validation MSE vs. Regularization Strength α\n'
             'House Price Prediction — 20 Features (5 Real, 15 Noise)',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=10, loc='upper left')
plt.tight_layout()
plt.show()

print(f'Optimal α: {best_alpha_mse:.4f}')
print(f'Minimum validation MSE: {best_val_mse:.2f}')
print(f'Corresponding RMSE: ${best_val_mse**0.5:.1f}k')

In [ ]:
# ---------------------------------------------------------------------------
# Section 11: Final Summary — Lasso vs Ridge vs OLS Comparison
# ---------------------------------------------------------------------------
from sklearn.linear_model import Ridge

best_lasso  = Lasso(alpha=best_alpha_mse, fit_intercept=True, max_iter=10000)
best_ridge  = Ridge(alpha=best_alpha_mse * 100)   # Ridge needs larger alpha typically
ols_model   = LinearRegression(fit_intercept=True)

for name, model in [('OLS', ols_model), ('Ridge', best_ridge), ('Lasso', best_lasso)]:
    model.fit(X_train, y_train)

models = {'OLS': ols_model, 'Ridge': best_ridge, 'Lasso': best_lasso}

fig, axes = plt.subplots(1, 3, figsize=(16, 6))

for ax, (name, model) in zip(axes, models.items()):
    coefs = model.coef_
    real_coefs  = coefs[:N_REAL]
    noise_coefs = coefs[N_REAL:]

    x_pos = np.arange(N_FEAT)
    bar_cols = [COLORS['real']]*N_REAL + [COLORS['noise']]*N_NOISE
    ax.bar(x_pos, coefs, color=bar_cols, edgecolor='white', alpha=0.85)
    ax.axhline(0, color='black', linewidth=1.0)

    n_nz = np.sum(np.abs(coefs) > 1e-4)
    val_mse_final = mean_squared_error(y_val, model.predict(X_val))

    ax.set_title(f'{name}\nNon-zero coeffs: {n_nz}/{N_FEAT}  |  Val RMSE: ${val_mse_final**0.5:.1f}k',
                 fontsize=12, fontweight='bold')
    ax.set_xlabel('Feature Index', fontsize=11)
    ax.set_ylabel('Coefficient', fontsize=11)

    # Separator line between real and noise
    ax.axvline(N_REAL - 0.5, color='black', linewidth=1.5, linestyle='--', alpha=0.5)

    real_patch  = mpatches.Patch(color=COLORS['real'],  label='Real features')
    noise_patch = mpatches.Patch(color=COLORS['noise'], label='Noise features')
    ax.legend(handles=[real_patch, noise_patch], fontsize=9)

plt.suptitle('OLS vs Ridge vs Lasso: Coefficient Sparsity and Prediction Performance',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('Comparison summary:')
print(f'{"Model":<10} {"Non-zero":>10} {"Train RMSE":>12} {"Val RMSE":>12}')
print('-' * 48)
for name, model in models.items():
    nz  = np.sum(np.abs(model.coef_) > 1e-4)
    tr  = mean_squared_error(y_train, model.predict(X_train))**0.5
    vl  = mean_squared_error(y_val,   model.predict(X_val  ))**0.5
    print(f'{name:<10} {nz:>10} {tr:>12.2f} {vl:>12.2f}')

## Section 12: Summary & Key Takeaways

### The L1 Objective
$$\text{Lasso Loss} = \frac{1}{2n}\sum_i(y_i - \hat{y}_i)^2 + \alpha\sum_j|\theta_j|$$

### Why Lasso Creates Sparsity — Three Perspectives

| Perspective | Explanation |
|-------------|-------------|
| **Geometric** | L1 constraint region is a diamond with corners on coordinate axes. The OLS loss ellipse almost always touches a corner first, zeroing one coordinate. |
| **Algorithmic** | Soft-thresholding has a "dead zone" $[-\alpha, +\alpha]$ where the optimal update is exactly 0. Small correlations fall into this zone. |
| **Bayesian** | L1 penalty is equivalent to a Laplace prior on coefficients (sharp peak at 0). This prior strongly favors sparse solutions. |

### When to Use Lasso
- **Sparse signals:** You believe only a small fraction of features matter
- **p >> n:** More features than samples (OLS is undefined; Ridge works but keeps all features)
- **Interpretability:** You want a model with few features that are easy to explain
- **Automated feature selection:** Let the model find relevant features during training

### When Ridge May Be Better
- Correlated features: L1 arbitrarily picks one among correlated features; Ridge distributes across them
- All features are relevant: No sparsity needed, Ridge's smoother shrinkage works better
- Numerical stability: Ridge is always numerically stable; Lasso can oscillate with high correlation

### Elastic Net: Best of Both
$$\text{Elastic Net} = \frac{1}{2n}\sum_i(y_i - \hat{y}_i)^2 + \alpha[\rho\sum_j|\theta_j| + (1-\rho)\sum_j\theta_j^2]$$
The `l1_ratio` $\rho$ controls the mix: $\rho=1$ → Lasso, $\rho=0$ → Ridge.

## Self-Check Questions

Test your understanding. Try to answer each question before expanding the solution.

---

**Question 1:**  
Lasso sets small coefficients to **exactly zero**. Ridge makes them very small but **never zero**. Why is this? (Hint: think about the geometry of the constraint regions and the derivative of the penalty.)

<details>
<summary>Click to reveal answer</summary>

**The answer comes from two complementary perspectives:**

**Geometric explanation:**  
L1 regularization constrains the solution to lie within the region $\sum_j |\theta_j| \leq t$ (a diamond in 2D). This region has **sharp corners on the coordinate axes**. When you expand the OLS error ellipse outward from the minimum, it almost always makes first contact with one of these corners — and at a corner, one (or more) coordinates is exactly zero.

The L2 region $\sum_j \theta_j^2 \leq t$ is a circle (sphere in higher dimensions) — **smooth, with no corners**. The expanding ellipse touches the circle at a generic point away from the axes. Both coordinates are non-zero at that contact point.

**Calculus/subgradient explanation:**  
For Ridge, the penalty is $\theta^2$, whose derivative is $2\theta$. At $\theta = 0$, the derivative is 0, so there's no force pushing the solution to stay at zero — any small perturbation moves it away.

For Lasso, the penalty is $|\theta|$, which is **not differentiable at $\theta = 0$**. Instead, it has a subdifferential $[-1, +1]$ at zero. The optimality condition at $\theta_j = 0$ is satisfied whenever $\rho_j \in [-\alpha, +\alpha]$ — there's a range of inputs that all map to the same output of zero. This is the "dead zone" in soft-thresholding. Small correlations (signals) get trapped in this dead zone and stay at zero.

**Intuition:** Ridge asks "how much should I shrink θ?" and always gives a non-zero answer. Lasso asks "should I include this feature at all?" and says "no" whenever the signal is weaker than the threshold α.
</details>

---

**Question 2:**  
You have 1,000 features and only 200 training samples (p >> n). Why might Lasso be a better starting point than Ridge?

<details>
<summary>Click to reveal answer</summary>

**Three key reasons:**

**1. OLS is undefined (p > n):**  
With 1,000 features and 200 samples, the design matrix X is 200×1,000 — it is not full column rank. The normal equations $(X^TX)\theta = X^Ty$ have infinitely many solutions. Both Ridge and Lasso add a penalty that makes the problem well-defined, but OLS cannot be used at all.

**2. Sparsity as domain knowledge:**  
In most real-world settings with 1,000 features, the truth is that most features are irrelevant. Lasso's implicit assumption of sparsity (most coefficients should be zero) is appropriate. Ridge keeps all 1,000 features in the model with small but non-zero weights — this is harder to interpret and wastes capacity on noise.

**3. Effective degrees of freedom:**  
Ridge with 200 samples and 1,000 parameters still has very high effective degrees of freedom (unless α is huge). Lasso naturally limits the effective model complexity to the number of non-zero coefficients, which is much less than 1,000.

**4. Predictive performance:**  
If the true data-generating process uses k << p features, Lasso's sparse solution will have lower variance than Ridge. The bias-variance tradeoff favors the sparser model.

**Practical advice:**  
- Start with Lasso to identify relevant features
- Then refit OLS or Ridge on just those selected features
- Or use Elastic Net if features are correlated (e.g., gene expression data)
</details>

---

**Question 3:**  
Coordinate descent updates one coefficient at a time. Why does it converge to the **global minimum** for Lasso (but might not for general problems)?

<details>
<summary>Click to reveal answer</summary>

**The key condition is convexity.**

**Why coordinate descent can fail in general:**  
For non-convex functions (e.g., neural network loss with ReLU activations), coordinate descent can get stuck at a saddle point or local minimum that is not globally optimal. When you minimize over one coordinate while holding others fixed, you can reach a point where each individual coordinate is at its optimum, but the combined point is not the global minimum.

**Why Lasso is different — it's convex:**  
The Lasso objective is the sum of:
1. **Squared loss:** $\frac{1}{2n}\|y - X\theta\|^2$ — this is a convex function (it's a sum of squared linear functions)
2. **L1 penalty:** $\alpha\sum_j|\theta_j|$ — this is also convex (the absolute value function is convex)

The sum of convex functions is convex. A convex function has only one global minimum (or a convex set of minima). For a convex function, if each coordinate is at its minimum given the others, then you've found the global minimum — this is a theorem from convex analysis.

**Additional technical note:**  
Coordinate descent for Lasso is guaranteed to converge because:
1. The function is convex in each coordinate separately (even though the L1 term is non-smooth)
2. The soft-thresholding step gives the **exact** coordinate-wise minimum, not just a gradient step
3. The objective strictly decreases at each update (unless already at the optimum)

This exact coordinate-wise solution (via soft-thresholding) is what makes coordinate descent efficient for Lasso — each update moves directly to the coordinate minimum rather than taking a small step toward it.
</details>